# Cleaning the age groups and sex dataset

This notebook inspects and cleans the official Latvian migration dataset `ibe030-migration-by-age-groups-and-sex-2016-2025-raw.csv`.
The goal is to inspect the raw data, convert the wide format year columns into a long format dataset, clean the dataset, validate the result after melting and cleaning, and save the dataset in `data/interim` folder for subsequent transformation.

In [1]:
import pandas as pd

age_groups_and_sex_data = pd.read_csv("../data/raw/ibe030-migration-by-age-groups-and-sex-2016-2025-raw.csv", skiprows=2)

age_groups_and_sex_data.head()

,Rādītāji,Vecuma grupa,Dzimums,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Iebrauca Latvijā,Pavisam,Pavisam,8345,9916,10909,11223,8840,12689,38708,15629,14701,13826
1,Iebrauca Latvijā,Pavisam,Vīrieši,4740,6376,7435,7664,5783,8399,19152,8951,8029,7756
2,Iebrauca Latvijā,Pavisam,Sievietes,3605,3540,3474,3559,3057,4290,19556,6678,6672,6070
3,Iebrauca Latvijā,0–4 gadi,Pavisam,976,747,670,655,484,507,1918,723,670,676
4,Iebrauca Latvijā,0–4 gadi,Vīrieši,538,428,377,357,278,287,985,380,336,369


The required libraries are imported and the raw age groups and sex dataset is loaded.
The first two rows of the `.csv` file contain the dataset title and a blank line before the actual column names, therefore, `skiprows=2` is used so that pandas reads the correct header row.
The first rows confirm that the dataset was loaded with the expected identifier columns, `Rādītāji`, `Vecuma grupa` and `Dzimums`, and the year columns from 2016 to 2025.

In [2]:
age_groups_and_sex_data.shape

(102, 13)

The dataset contains 102 rows and 13 columns.
The rows represent two migration indicators for 17 age groups, and three sex categories. The columns consist of three identifier columns, `Rādītāji`, `Vecuma grupa` and `Dzimums`, and ten year columns from 2016 to 2025.
The shape, together with the displayed first rows, confirms that the dataset is structured in a wide format.

In [3]:
age_groups_and_sex_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rādītāji      102 non-null    str  
 1   Vecuma grupa  102 non-null    str  
 2   Dzimums       102 non-null    str  
 3   2016          102 non-null    int64
 4   2017          102 non-null    int64
 5   2018          102 non-null    int64
 6   2019          102 non-null    int64
 7   2020          102 non-null    int64
 8   2021          102 non-null    int64
 9   2022          102 non-null    int64
 10  2023          102 non-null    int64
 11  2024          102 non-null    int64
 12  2025          102 non-null    int64
dtypes: int64(10), str(3)
memory usage: 10.5 KB


The dataset contains 102 non-null entries for every column.
`Rādītāji`, `Vecuma grupa` and `Dzimums` use pandas default `str` data type. The ten your columns use `int64` data type.
This confirms that all required year columns are present, all values in the year columns are numeric, and there are no missing values in the dataset.

In [4]:
identifier_columns = ["Rādītāji", "Vecuma grupa", "Dzimums"]

for column in identifier_columns:
    print(age_groups_and_sex_data[column].value_counts(dropna=False))

Rādītāji
Iebrauca Latvijā        51
Izbrauca no Latvijas    51
Name: count, dtype: int64
Vecuma grupa
Pavisam              6
0–4 gadi             6
5–9 gadi             6
10–14 gadi           6
15–19 gadi           6
20–24 gadi           6
25–29 gadi           6
30–34 gadi           6
35–39 gadi           6
40–44 gadi           6
45–49 gadi           6
50–54 gadi           6
55–59 gadi           6
60–64 gadi           6
65–69 gadi           6
70–74 gadi           6
75 gadi un vairāk    6
Name: count, dtype: int64
Dzimums
Pavisam      34
Vīrieši      34
Sievietes    34
Name: count, dtype: int64


The identifier columns contain only the expected values:
- `Rādītāji` contains two migration identifiers: `Iebrauca Latvijā` and `Izbrauca no Latvijas`. Each appears 51 times, once for each combination of age group and sex.
- `Vecuma grupa` contains 17 age groups. Each appears six times, once for each sex category.
- `Dzimums` contains three categories: `Pavisam`, `Vīrieši` and `Sievietes`. Each appears 34 times, once for each combination of migration identifier and age group.

In [5]:
print("Count of missing values:")
print(age_groups_and_sex_data.isna().sum())

print("Count of empty strings:")
print(age_groups_and_sex_data[identifier_columns].eq("").sum())

Count of missing values:
Rādītāji        0
Vecuma grupa    0
Dzimums         0
2016            0
2017            0
2018            0
2019            0
2020            0
2021            0
2022            0
2023            0
2024            0
2025            0
dtype: int64
Count of empty strings:
Rādītāji        0
Vecuma grupa    0
Dzimums         0
dtype: int64


The dataset does not contain missing values in any column. The identifier columns do not contain empty strings either.
This confirms that no missing value handling or empty string handling is required.

In [6]:
for column in identifier_columns:
    stripped_values = age_groups_and_sex_data[column].astype("string").str.strip()

    whitespace_count = stripped_values.ne(age_groups_and_sex_data[column].astype("string")).sum()

    print(f"{column}: {whitespace_count} values with leading or trailing whitespace.")

Rādītāji: 0 values with leading or trailing whitespace.
Vecuma grupa: 0 values with leading or trailing whitespace.
Dzimums: 0 values with leading or trailing whitespace.


The identifier columns are checked for leading or trailing whitespace, which may not be visible when the values are displayed.
For each identifier column, the values are stripped of leading and trailing whitespace and compared with the original values.
Bot identifier columns contain no leading or trailing whitespace, therefore, no whitespace cleaning is required.

In [7]:
age_groups_and_sex_data[identifier_columns].duplicated().sum()

np.int64(0)

The identifier columns are checked for duplicate entries in the dataset.
Result of `0` confirms that no duplicates are present. No duplicate rows need to be removed.

In [8]:
year_columns = age_groups_and_sex_data.columns[3:]

age_groups_and_sex_melted = age_groups_and_sex_data.melt(
    id_vars=identifier_columns,
    value_vars=year_columns,
    var_name="Gads",
    value_name="Skaits"
)

age_groups_and_sex_melted.head()

,Rādītāji,Vecuma grupa,Dzimums,Gads,Skaits
0,Iebrauca Latvijā,Pavisam,Pavisam,2016,8345
1,Iebrauca Latvijā,Pavisam,Vīrieši,2016,4740
2,Iebrauca Latvijā,Pavisam,Sievietes,2016,3605
3,Iebrauca Latvijā,0–4 gadi,Pavisam,2016,976
4,Iebrauca Latvijā,0–4 gadi,Vīrieši,2016,538


The dataset is converted from wide format to long format.
`year_columns` selects the ten year columns. The `.melt()` method keeps the identifier columns, and converts each year column into a row. The former year column names are stored in `Gads`, and their published migration counts are stored in `Skaits`.
The resulting dataset contains one row for every migration indicator, age group, sex and year.

In [9]:
age_groups_and_sex_melted["Gads"] = age_groups_and_sex_melted["Gads"].astype("int64")

After melting, `Gads` column contains former year column names. During melting, these column names are stored as strings, so `Gads` has the `str` data type.
`Gads` is converted to `int64`, matching the data type of the year columns in the original dataset. This keeps the years as numeric, ordered values for later transformations, such as filtering or comparing values by year, if necessary.

In [10]:
age_groups_and_sex_melted.shape

(1020, 5)

The melted dataset contains 1020 rows and 5 columns.
The rows represent every combination of two migrations indicators, 17 age groups, three sex categories and ten years.
The five columns now are `Rādītāji`, `Vecuma grupa`, `Dzimums`, `Gads`, and `Skaits`.
This confirms that the melting from wide to long format has been successful.

In [11]:
age_groups_and_sex_melted.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Rādītāji      1020 non-null   str  
 1   Vecuma grupa  1020 non-null   str  
 2   Dzimums       1020 non-null   str  
 3   Gads          1020 non-null   int64
 4   Skaits        1020 non-null   int64
dtypes: int64(2), str(3)
memory usage: 40.0 KB


The melted dataset, just as the original, contains no null values in its 1020 rows.
`Rādītāji`, `Vecuma grupa`, `Dzimums` keep `str` data type as intended. `Gads` and `Skaits` use `int64` data type.
This confirms that the year conversion has been successful, and that `Gads` conversion has also succeeded.

In [12]:
print("All counts are non-negative:")
print((age_groups_and_sex_melted["Skaits"] >= 0).all())

print("All counts are whole numbers:")
print((age_groups_and_sex_melted["Skaits"] % 1 == 0).all())

All counts are non-negative:
True
All counts are whole numbers:
True


The cleaned migration counts are checked to confirm that they are valid. `Skaits >= 0` checks that every published count is non-negative. `Skaits % 1 == 0` checks that every count is a whole number.
Both checks return `True`, confirming that no invalid values were introduced by accident during cleaning.

In [13]:
checks_to_compare = [
    ("Iebrauca Latvijā", "Pavisam", "Pavisam", 2016, 8345),
    ("Iebrauca Latvijā", "Pavisam", "Pavisam", 2025, 13826),
    ("Izbrauca no Latvijas", "Pavisam", "Pavisam", 2016, 20574),
    ("Izbrauca no Latvijas", "Pavisam", "Pavisam", 2025, 15117)
]

for indicator, age_group, sex, year, expected_count in checks_to_compare:
    actual_count = age_groups_and_sex_melted.loc[
        (age_groups_and_sex_melted["Rādītāji"] == indicator)
        & (age_groups_and_sex_melted["Vecuma grupa"] == age_group)
        & (age_groups_and_sex_melted["Dzimums"] == sex)
        & (age_groups_and_sex_melted["Gads"] == year),
        "Skaits"
    ].item()

    print(f"{indicator} | {age_group} | {sex} | {year}: expected {expected_count}, found {actual_count}")

Iebrauca Latvijā | Pavisam | Pavisam | 2016: expected 8345, found 8345
Iebrauca Latvijā | Pavisam | Pavisam | 2025: expected 13826, found 13826
Izbrauca no Latvijas | Pavisam | Pavisam | 2016: expected 20574, found 20574
Izbrauca no Latvijas | Pavisam | Pavisam | 2025: expected 15117, found 15117


Selected rows are checked to confirm that they still match the correct migration flow, age group, sex and year after melting.
For each check, the expected count is taken from the originally loaded dataset. The melted dataset is filtered by migration indicator, age group, sex and year, and the resulting `Skaits` value is compared with the expected count.
All selected values match, confirming that the melting preserved the correct migration flows, age groups, sexes, years and published counts.

In [14]:
age_groups_and_sex_melted.to_csv("../data/interim/ibe030-migration-by-age-groups-and-sex-2016-2025-interim.csv", index=False, encoding="utf-8-sig")
print("The dataset has been saved")

The dataset has been saved


The cleaned long format dataset is saved to the `data/interim` folder for use in transformations as the next step in the planned methodology.
`index=False` prevents pandas from writing its numeric row index as an extra column in the `.csv` file. `encoding="utf-8-sig"` writes a UTF-8 file with a bite order mark, which helps some programs, including Microsoft Excel, to correctly recognize the encoding and display Latvian characters.